# 02 — Training (9-class Nature Classifier)

Trains a custom ResNet-style network to classify nature element photos into 9 classes.
Produces `best_model.pth` — used by `03_evaluation_export.ipynb` for ONNX conversion.

**Datasets required — add via Kaggle → Add Data:**
- `noahbadoa/plantnet-300k-images`
- `benjaminkz/places365`

**Accelerator: GPU T4 x2** (Settings → Accelerator).  
**Expected runtime:** ~2.5–3.5 hours for the full 30-epoch run.

The first batch of Phase A takes 30–60 s while `torch.compile` JIT-compiles the model — this is normal.

In [ ]:
import os, json, random, collections, pathlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch:  {torch.__version__}')
print(f'Device:   {device}')
if device.type == 'cuda':
    print(f'GPU:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cudnn.benchmark = True
else:
    print('WARNING: No GPU. Training will be very slow.')

# Cache dir for torch.compile on Kaggle
if os.path.exists('/kaggle/working'):
    os.environ['TORCHINDUCTOR_CACHE_DIR'] = '/kaggle/working/torch_cache'

# ── Constants ─────────────────────────────────────────────────────────────────
CLASSES = ['not_nature','tree','shrub','grass_lawn','mulch',
           'garden_bed','ground_cover','green_roof','water_body']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES   = len(CLASSES)
TARGET_PER_CLASS = 5_000
BATCH_SIZE    = 32
CHECKPOINT    = '/kaggle/working/best_model.pth'

## Part 1 — Data collection

Regenerates `train.csv / val.csv / test.csv` with paths valid for this environment.
Identical logic to `01_data_preparation.ipynb` — takes ~60 s.

In [ ]:
# ── Locate datasets ───────────────────────────────────────────────────────────
def _find_dataset(keyword):
    root = pathlib.Path("/kaggle/input")
    for d1 in root.iterdir():
        if not d1.is_dir(): continue
        if keyword in d1.name.lower(): return d1
        for d2 in d1.iterdir():
            if not d2.is_dir(): continue
            if keyword in d2.name.lower(): return d2
            for d3 in d2.iterdir():
                if d3.is_dir() and keyword in d3.name.lower(): return d3
    return None

PLACES_ROOT   = _find_dataset("places")
PLANTNET_ROOT = _find_dataset("plantnet")
assert PLACES_ROOT is not None,   "Places365 not found — add benjaminkz/places365 via Add Data"
assert PLANTNET_ROOT is not None, "PlantNet not found — add noahbadoa/plantnet-300k-images via Add Data"
inner = PLANTNET_ROOT / "plantnet_300K"
if inner.exists(): PLANTNET_ROOT = inner
print(f"Places365:  {PLACES_ROOT}")
print(f"PlantNet:   {PLANTNET_ROOT}")

# ── Places365 helpers ─────────────────────────────────────────────────────────
def _load_places_category_counts(places_root):
    for fname in ("train.txt", "val.txt"):
        txt = pathlib.Path(places_root) / fname
        if not txt.exists(): continue
        counts = collections.Counter()
        with open(txt) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rel = line.split()[0]
                cat = pathlib.Path(rel).parent.name
                counts[cat] += 1
        return dict(counts), fname
    return {}, None

def _find_category_dir(places_root, category_name):
    places_root = pathlib.Path(places_root)
    for sub in ("train", "val", "data_256", "data_large"):
        base = places_root / sub
        if not base.exists(): continue
        if (base / category_name).is_dir(): return base / category_name
        nested = base / category_name[0].lower() / category_name
        if nested.is_dir(): return nested
    return None

IMAGE_EXTS = (".jpg", ".jpeg", ".png")

def collect_from_places365(include=None, exclude=None, target=5000):
    if target <= 0: return []
    matching = [
        cat for cat in PLACES_CATS
        if (not exclude or not any(kw in cat.lower() for kw in exclude))
        and (not include or any(kw in cat.lower() for kw in include))
    ]
    random.shuffle(matching)
    paths = []
    for cat in matching:
        cat_dir = _find_category_dir(PLACES_ROOT, cat)
        if cat_dir is None: continue
        imgs = [str(cat_dir / f) for f in os.listdir(cat_dir)
                if f.lower().endswith(IMAGE_EXTS)]
        paths.extend(imgs)
        if len(paths) >= target: break
    random.shuffle(paths)
    return paths[:target]

print("Parsing Places365 train.txt (~5 s)...")
PLACES_CATS, _src = _load_places_category_counts(PLACES_ROOT)
print(f"  {len(PLACES_CATS):,} categories from {_src}")

# ── PlantNet taxonomy ─────────────────────────────────────────────────────────
json_path = next(
    (p for p in pathlib.Path(_find_dataset("plantnet")).rglob("*.json")
     if "species" in p.name.lower() and "name" in p.name.lower()),
    None,
)
assert json_path is not None, "Species metadata JSON not found under PlantNet dataset"
with open(json_path) as f:
    species_names = json.load(f)

CATEGORY_GENERA = {
    "tree": {"quercus","fagus","betula","acer","fraxinus","tilia","ulmus","pinus",
             "abies","picea","larix","cedrus","taxus","populus","alnus","carpinus",
             "platanus","robinia","castanea","sorbus","liriodendron","metasequoia",
             "nothofagus","tsuga","thuja","pseudotsuga","liquidambar","gleditsia",
             "catalpa","gymnocladus","celtis"},
    "shrub": {"rosa","berberis","cornus","viburnum","buddleja","ligustrum","lonicera",
              "sambucus","cotoneaster","crataegus","ilex","buxus","lavandula","salvia",
              "cytisus","ulex","calluna","erica","daphne","nandina","pieris",
              "vaccinium","rubus","amelanchier","physocarpus","spiraea","syringa",
              "hydrangea"},
    "ground_cover": {"hedera","vinca","ajuga","pachysandra","hypericum","sedum",
                     "thymus","fragaria","potentilla","trifolium","lamium"},
    "grass_lawn": {"festuca","poa","lolium","agrostis","dactylis","holcus","bromus",
                   "brachypodium","arrhenatherum","phleum"},
}

def assign_category(species_name):
    genus = species_name.split("_")[0].lower()
    for category, genera in CATEGORY_GENERA.items():
        if genus in genera: return category
    return None

images_train_dir = PLANTNET_ROOT / "images_train"
assert images_train_dir.exists(), f"images_train not found at {images_train_dir}"

categorised = collections.defaultdict(list)
for species_id, species_name in species_names.items():
    if not isinstance(species_name, str): continue
    cat = assign_category(species_name)
    if cat is None: continue
    species_dir = images_train_dir / str(species_id)
    if species_dir.exists():
        categorised[cat].extend([str(p) for p in species_dir.glob("*.jpg")])

_tree_gap = max(0, TARGET_PER_CLASS - len(categorised["tree"]))
if _tree_gap > 0:
    categorised["tree"].extend(
        collect_from_places365(include=["tree_farm"], target=_tree_gap))

NATURE_KEYWORDS = [
    "forest","park","garden","beach","mountain","field","meadow","lake","river",
    "ocean","coast","cliff","valley","jungle","swamp","waterfall","glacier",
    "desert","woodland","nature","grass","lawn","greenhouse","nursery","wetland",
    "bog","pond","pasture","marsh","tundra","creek","lagoon","canal","orchard",
    "vineyard","golf","campsite","tree_farm","rice_paddy","moat",
    "swimming_hole","hot_spring","watering_hole",
]

not_nature_paths = collect_from_places365(exclude=NATURE_KEYWORDS, target=TARGET_PER_CLASS)
water_paths      = collect_from_places365(
    include=["lake","river","pond","ocean","coast","reservoir","canal","waterfall",
             "fishpond","creek","lagoon","marsh","moat","swimming_hole","hot_spring",
             "watering_hole","wave","underwater"],
    target=TARGET_PER_CLASS)
grass_shortfall  = max(0, TARGET_PER_CLASS - len(categorised["grass_lawn"]))
grass_supplement = collect_from_places365(
    include=["lawn","golf_course","pasture","athletic_field"], target=grass_shortfall)
garden_paths     = collect_from_places365(
    include=["formal_garden","botanical_garden","flower_garden","rose_garden",
             "japanese_garden","vegetable_garden"],
    target=TARGET_PER_CLASS)
green_roof_paths = collect_from_places365(
    include=["roof_garden","rooftop"], target=TARGET_PER_CLASS)
mulch_paths      = collect_from_places365(
    include=["topiary_garden","zen_garden","courtyard"], target=TARGET_PER_CLASS)

all_sources = {
    "not_nature":   not_nature_paths,
    "tree":         categorised["tree"],
    "shrub":        categorised["shrub"],
    "grass_lawn":   categorised["grass_lawn"] + grass_supplement,
    "mulch":        mulch_paths,
    "garden_bed":   garden_paths,
    "ground_cover": categorised["ground_cover"],
    "green_roof":   green_roof_paths,
    "water_body":   water_paths,
}

rows = []
for class_name, paths in all_sources.items():
    random.shuffle(paths)
    sample = paths[:TARGET_PER_CLASS]
    label  = CLASS_TO_IDX[class_name]
    for p in sample:
        rows.append((p, class_name, label))

df = pd.DataFrame(rows, columns=["path", "class_name", "label"])
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df   = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

train_df.to_csv("/kaggle/working/train.csv", index=False)
val_df  .to_csv("/kaggle/working/val.csv",   index=False)
test_df .to_csv("/kaggle/working/test.csv",  index=False)

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
print(train_df["class_name"].value_counts().reindex(CLASSES).to_string())
print("CSVs saved to /kaggle/working/")


## Part 2 — Dataset & DataLoader

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    # RandomErasing must come AFTER ToTensor — it operates on tensors, not PIL Images.
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# ── Dataset ───────────────────────────────────────────────────────────────────
class NatureDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        MAX_RETRIES = 5
        for attempt in range(MAX_RETRIES):
            try:
                img = Image.open(row['path']).convert('RGB')
                break
            except Exception:
                if attempt == MAX_RETRIES - 1:
                    img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
                else:
                    idx = random.randint(0, len(self) - 1)
                    row = self.df.iloc[idx]
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])

def _worker_init(worker_id):
    np.random.seed((torch.initial_seed() % (2**32)) + worker_id)
    random.seed((torch.initial_seed() % (2**32)) + worker_id)

# ── DataLoaders ───────────────────────────────────────────────────────────────
_pin = device.type == 'cuda'
train_loader = DataLoader(NatureDataset('/kaggle/working/train.csv', train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=_pin,
    worker_init_fn=_worker_init)
val_loader   = DataLoader(NatureDataset('/kaggle/working/val.csv',   val_transform),
    batch_size=64, shuffle=False, num_workers=2, pin_memory=_pin)
test_loader  = DataLoader(NatureDataset('/kaggle/working/test.csv',  val_transform),
    batch_size=64, shuffle=False, num_workers=2, pin_memory=_pin)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

# Quick shape check
imgs, labels = next(iter(train_loader))
print(f'Batch shape:  {imgs.shape}  labels: {labels.shape}  range: [{imgs.min():.2f}, {imgs.max():.2f}]')
print(f'Label counts: {dict(zip(*np.unique(labels.numpy(), return_counts=True)))}')


## Part 3 — Model

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=False)  # inplace=False — shortcut backward safety
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.shortcut = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                          nn.BatchNorm2d(out_ch))
            if stride != 1 or in_ch != out_ch else nn.Identity()
        )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))


class NatureResNet(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),  # 224→112
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                    # 112→56
        )
        self.stage1 = nn.Sequential(ResidualBlock(32,  32),  ResidualBlock(32,  32))
        self.stage2 = nn.Sequential(ResidualBlock(32,  64, 2), ResidualBlock(64,  64))
        self.stage3 = nn.Sequential(ResidualBlock(64,  128, 2), ResidualBlock(128, 128))
        self.stage4 = nn.Sequential(ResidualBlock(128, 256, 2), ResidualBlock(256, 256))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.stage2(x)
        x = self.stage3(x); x = self.stage4(x)
        return self.classifier(self.gap(x))


raw_model = NatureResNet(NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in raw_model.parameters())
print(f'Parameters: {total_params:,}  (expected ~2,800,000)')

# Shape check
with torch.no_grad():
    out = raw_model(torch.randn(4, 3, 224, 224, device=device))
assert out.shape == (4, NUM_CLASSES), f'Unexpected output shape: {out.shape}'
print(f'Output shape: {out.shape}  ✓')

# Compile for 1.2–1.5× throughput (adds ~60 s overhead on first batch)
model = torch.compile(raw_model)
print('torch.compile applied')

## Part 4 — Training loop functions

In [ ]:
criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler    = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
writer    = SummaryWriter('/kaggle/working/runs/nature_resnet')


def train_one_epoch(model, loader, criterion, optimiser, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimiser.zero_grad()
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss    = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimiser)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimiser)
            scaler.update()
        else:
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            preds   = outputs.argmax(1)
            total_loss += loss.item() * images.size(0)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds .extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


def save_checkpoint(epoch, phase, val_loss, val_acc):
    torch.save({
        'epoch': epoch, 'phase': phase,
        'model_state_dict':     raw_model.state_dict(),
        'optimiser_state_dict': optimiser.state_dict(),
        'val_loss': val_loss, 'val_acc': val_acc,
    }, CHECKPOINT)


print('Training functions ready.')

## Part 5 — Phase A  (lr = 1e-3, up to 20 epochs)

**First batch takes 30–60 s** while `torch.compile` JIT-compiles — this is normal, subsequent batches are fast.

Watch for:
- `val_loss` falling each epoch → learning
- `val_loss` rising while `train_loss` falls → overfitting (increase dropout / weight decay)
- Both flat after epoch 3 → underfitting (increase LR or add layers)

In [ ]:
scheduler_a   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode='min', patience=3, factor=0.5, min_lr=1e-6)
best_val_loss = float('inf')
no_improve    = 0
PATIENCE      = 5
EPOCHS_A      = 20

phase_a_history = []

print(f'{'Ep':>4}  {'train_loss':>10}  {'val_loss':>9}  '
      f'{'train_acc':>9}  {'val_acc':>8}  {'lr':>9}  notes')
print('-' * 72)

for epoch in range(1, EPOCHS_A + 1):
    t_start = time.time()
    train_loss, train_acc         = train_one_epoch(model, train_loader, criterion, optimiser, scaler, device)
    val_loss,   val_acc, _, _     = evaluate(model, val_loader, criterion, device)
    scheduler_a.step(val_loss)
    lr = optimiser.param_groups[0]['lr']
    elapsed = time.time() - t_start

    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Loss/val',   val_loss,   epoch)
    writer.add_scalar('Acc/train',  train_acc,  epoch)
    writer.add_scalar('Acc/val',    val_acc,    epoch)
    writer.add_scalar('LR',         lr,         epoch)

    note = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve    = 0
        save_checkpoint(epoch, 'A', val_loss, val_acc)
        note = '✓ saved'
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            note = f'early stop (patience={PATIENCE})'

    phase_a_history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                                train_acc=train_acc, val_acc=val_acc, lr=lr))
    print(f'{epoch:>4}  {train_loss:>10.4f}  {val_loss:>9.4f}  '
          f'{train_acc:>9.3f}  {val_acc:>8.3f}  {lr:>9.2e}  {note}  [{elapsed:.0f}s]')

    if no_improve >= PATIENCE:
        break

print(f'\nPhase A complete.  Best val_loss: {best_val_loss:.4f}')

In [ ]:
# ── Phase A curves ────────────────────────────────────────────────────────────
h = pd.DataFrame(phase_a_history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(h.epoch, h.train_loss, label='train')
axes[0].plot(h.epoch, h.val_loss,   label='val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(h.epoch, h.train_acc, label='train')
axes[1].plot(h.epoch, h.val_acc,   label='val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(h.epoch, h.lr)
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')

plt.suptitle('Phase A training curves')
plt.tight_layout()
plt.show()

# Overfitting check
gap = h.train_acc.iloc[-1] - h.val_acc.iloc[-1]
print(f'Final train/val acc gap: {gap:.3f}')
if gap > 0.15:
    print('⚠  Large gap — model is overfitting. Consider increasing dropout or weight_decay.')
else:
    print('✓  Gap within acceptable range.')

## Part 6 — Phase B  (lr = 1e-4, up to 10 epochs)

Loads the best Phase A checkpoint and fine-tunes at a lower learning rate.
The Phase A adaptive moment estimates are preserved — only the LR is reset.

In [ ]:
# Load best Phase A checkpoint
ckpt = torch.load(CHECKPOINT, map_location=device)
raw_model.load_state_dict(ckpt['model_state_dict'])
optimiser.load_state_dict(ckpt['optimiser_state_dict'])

# Override LR — load_state_dict restores Phase A lr
for g in optimiser.param_groups:
    g['lr'] = 1e-4

# Reinitialise scheduler — clears Phase A internal best_loss state
scheduler_b = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode='min', patience=3, factor=0.5, min_lr=1e-6)

best_val_loss = ckpt['val_loss']   # continue improving from Phase A's best
no_improve    = 0
EPOCHS_B      = 10
phase_b_start = ckpt['epoch']      # for TensorBoard continuity

phase_b_history = []

print(f'Loaded Phase A checkpoint (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')
print()
print(f'{'Ep':>4}  {'train_loss':>10}  {'val_loss':>9}  '
      f'{'train_acc':>9}  {'val_acc':>8}  {'lr':>9}  notes')
print('-' * 72)

for epoch in range(1, EPOCHS_B + 1):
    global_epoch = phase_b_start + epoch
    t_start = time.time()
    train_loss, train_acc     = train_one_epoch(model, train_loader, criterion, optimiser, scaler, device)
    val_loss,   val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler_b.step(val_loss)
    lr = optimiser.param_groups[0]['lr']
    elapsed = time.time() - t_start

    writer.add_scalar('Loss/train', train_loss, global_epoch)
    writer.add_scalar('Loss/val',   val_loss,   global_epoch)
    writer.add_scalar('Acc/train',  train_acc,  global_epoch)
    writer.add_scalar('Acc/val',    val_acc,    global_epoch)
    writer.add_scalar('LR',         lr,         global_epoch)

    note = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve    = 0
        save_checkpoint(global_epoch, 'B', val_loss, val_acc)
        note = '✓ saved'
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            note = f'early stop'

    phase_b_history.append(dict(epoch=global_epoch, train_loss=train_loss, val_loss=val_loss,
                                train_acc=train_acc, val_acc=val_acc, lr=lr))
    print(f'{epoch:>4}  {train_loss:>10.4f}  {val_loss:>9.4f}  '
          f'{train_acc:>9.3f}  {val_acc:>8.3f}  {lr:>9.2e}  {note}  [{elapsed:.0f}s]')

    if no_improve >= PATIENCE:
        break

writer.close()
print(f'\nPhase B complete.  Best val_loss: {best_val_loss:.4f}')
print(f'Checkpoint saved to: {CHECKPOINT}')

## Part 7 — Val set evaluation

Load the best checkpoint (may be Phase A or B) and report per-class metrics.
This is a quick sanity check — full evaluation + ONNX export is in `03_evaluation_export.ipynb`.

In [ ]:
# Reload best checkpoint for final eval
ckpt = torch.load(CHECKPOINT, map_location=device)
raw_model.load_state_dict(ckpt['model_state_dict'])
print(f'Evaluating checkpoint: epoch {ckpt["epoch"]}, phase {ckpt["phase"]}, '
      f'val_loss={ckpt["val_loss"]:.4f}, val_acc={ckpt["val_acc"]:.3f}')

_, _, val_preds, val_labels = evaluate(raw_model, val_loader, criterion, device)

print()
print(classification_report(val_labels, val_preds, target_names=CLASSES, zero_division=0))

# Combined training curve (both phases)
all_hist = pd.DataFrame(phase_a_history + phase_b_history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
vline = phase_b_start  # where Phase B started
for ax, col, title in [
        (axes[0], ('train_loss','val_loss'), 'Loss'),
        (axes[1], ('train_acc', 'val_acc'),  'Accuracy')]:
    ax.plot(all_hist.epoch, all_hist[col[0]], label='train')
    ax.plot(all_hist.epoch, all_hist[col[1]], label='val')
    ax.axvline(vline, color='gray', linestyle='--', label='Phase B')
    ax.set_title(title); ax.legend(); ax.set_xlabel('Epoch')
plt.suptitle('Full training history (A + B)')
plt.tight_layout()
plt.show()

print(f'\nDownload best_model.pth from the Output tab.')
print('Then run 03_evaluation_export.ipynb for ONNX conversion.')